# 01 — Colab data preparation

This notebook downloads the IEEE-CIS competition data using the Kaggle API, performs the required **left join**, creates shared features, and writes time-based train/validation/test splits. Run it once before the three model notebooks.

**Google Colab setup:** upload `kaggle.json` when prompted. You must have accepted the IEEE-CIS competition rules on Kaggle first.

In [ ]:
!pip -q install kaggle pandas numpy pyarrow

from google.colab import files
uploaded = files.upload()  # Upload kaggle.json downloaded from Kaggle Settings → API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!mkdir -p /content/ieee_fraud/raw /content/ieee_fraud/processed

In [ ]:
# Download (about 1.35 GB extracted). This requires Kaggle competition access.
!kaggle competitions download -c ieee-fraud-detection -p /content/ieee_fraud/raw
!unzip -oq /content/ieee_fraud/raw/ieee-fraud-detection.zip -d /content/ieee_fraud/raw
!ls -lh /content/ieee_fraud/raw

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

RAW = Path('/content/ieee_fraud/raw')
OUT = Path('/content/ieee_fraud/processed')
RANDOM_SEED = 42

train_tx = pd.read_csv(RAW / 'train_transaction.csv')
train_id = pd.read_csv(RAW / 'train_identity.csv')
test_tx = pd.read_csv(RAW / 'test_transaction.csv')
test_id = pd.read_csv(RAW / 'test_identity.csv')

# Left join is essential: only a subset of transactions have identity records.
train = train_tx.merge(train_id, on='TransactionID', how='left', indicator='identity_join')
test = test_tx.merge(test_id, on='TransactionID', how='left', indicator='identity_join')
train['has_identity'] = (train['identity_join'] == 'both').astype('int8')
test['has_identity'] = (test['identity_join'] == 'both').astype('int8')
train = train.drop(columns='identity_join')
test = test.drop(columns='identity_join')

assert len(train) == len(train_tx)
assert len(test) == len(test_tx)
print(train.shape, test.shape, 'fraud rate:', train.isFraud.mean())

In [ ]:
# Shared, leakage-safe row-level features. Do not calculate future-looking aggregates here.
def add_common_features(df):
    df = df.copy()
    df['log_TransactionAmt'] = np.log1p(df['TransactionAmt'].clip(lower=0))
    df['transaction_day'] = (df['TransactionDT'] // 86_400).astype('int16')
    df['transaction_week'] = (df['TransactionDT'] // (86_400 * 7)).astype('int16')
    df['transaction_hour'] = ((df['TransactionDT'] // 3_600) % 24).astype('int8')
    df['is_weekend'] = ((df['transaction_day'] % 7).isin([5, 6])).astype('int8')
    for col in ['TransactionAmt', 'dist1', 'dist2', 'DeviceInfo', 'id_30', 'id_31']:
        if col in df:
            df[f'{col}_missing'] = df[col].isna().astype('int8')
    for cols, name in [(['card1', 'card2'], 'card1_card2'), (['addr1', 'addr2'], 'addr1_addr2'), (['P_emaildomain', 'R_emaildomain'], 'email_pair')]:
        if set(cols).issubset(df.columns):
            df[name] = df[cols].fillna('MISSING').astype(str).agg('_'.join, axis=1)
    return df

train = add_common_features(train)
test = add_common_features(test)

# Remove completely empty or constant training features; apply the same removal to test.
feature_columns = [c for c in train.columns if c not in ['isFraud', 'TransactionID']]
drop_columns = [c for c in feature_columns if train[c].isna().all() or train[c].nunique(dropna=False) <= 1]
feature_columns = [c for c in feature_columns if c not in drop_columns]

# Chronological 70 / 15 / 15 split. Never use a random split for final evaluation.
ordered = train.sort_values('TransactionDT').reset_index(drop=True)
n = len(ordered)
train_end, validation_end = int(n * .70), int(n * .85)
train_split = ordered.iloc[:train_end]
validation_split = ordered.iloc[train_end:validation_end]
test_split = ordered.iloc[validation_end:]

for name, frame in [('train', train_split), ('validation', validation_split), ('test', test_split)]:
    frame.to_parquet(OUT / f'{name}.parquet', index=False)
test[feature_columns].to_parquet(OUT / 'kaggle_test_features.parquet', index=False)

metadata = {'feature_columns': feature_columns, 'dropped_columns': drop_columns, 'split_sizes': {'train': len(train_split), 'validation': len(validation_split), 'test': len(test_split)}}
(OUT / 'preparation_metadata.json').write_text(json.dumps(metadata, indent=2))
print(metadata['split_sizes'], 'features:', len(feature_columns), 'dropped:', len(drop_columns))

In [ ]:
# Optional: persist processed files and metadata to Drive for the next notebooks.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ieee_fraud
!cp -r /content/ieee_fraud/processed /content/drive/MyDrive/ieee_fraud/
print('Saved to Google Drive: MyDrive/ieee_fraud/processed')